# Penalties and degrees of freedom

Penalized learners are specified by a target **degrees of freedom** `df` rather
than a raw smoothing parameter $\lambda$. This chapter derives the mapping
from `df` to $\lambda$ and explains how it makes learner flexibility comparable.

## Degrees of freedom of a linear smoother

Let $X$ be the learner design, $W$ the diagonal matrix of observation weights,
and $K$ its penalty. Omit zero-weight rows and assume $X^\top W X$ is
nonsingular for the derivation below. Write $Z=W^{1/2}X$. The symmetric hat matrix

$$
H(\lambda)=Z(X^\top W X+\lambda K)^{-1}Z^\top
$$

maps the weighted response $W^{1/2}y$ to the weighted fitted values.
The degrees-of-freedom calculation uses this weighted-coordinate matrix;
the response-scale matrix $X(X^\top W X+\lambda K)^{-1}X^\top W$ is generally
not symmetric. The trace-based measure used by `mboost` is:

$$
\mathrm{df}(\lambda) = \operatorname{tr}\!\big(2 H(\lambda) - H(\lambda) H(\lambda)^\top\big).
$$

It is monotone decreasing in $\lambda$: at $\lambda = 0$ it equals
$\operatorname{rank}(X)$ (the unpenalized fit), and as $\lambda \to \infty$ it
falls to $\dim \ker K$ (the unpenalized null space of the penalty — e.g. 1 for a
ridge with an unpenalized intercept, 0 for a full ridge penalty, and 2 for a second-order difference penalty, which leaves a line unpenalized).

```{note}
An alternative, $\mathrm{df} = \operatorname{tr}(H)$, is also in use; `mboostrs`
follows `mboost`'s default $\operatorname{tr}(2H - HH^\top)$, so a given `df`
calibrates to the same $\lambda$ as in R (to $2 \times 10^{-7}$ relative on the
data set used for the comparison with R).
```

### Derivation: from the hat matrix to the eigenvalue sum

The root-find evaluates a closed form of $\mathrm{df}(\lambda)$, derived here.

In weighted coordinates, the hat matrix is

$$
H(\lambda)=W^{1/2}X(X^\top W X+\lambda K)^{-1}X^\top W^{1/2}.
$$

Write the penalised Gram matrix $G = X^\top W X + \lambda K$. The trace we
need is

$$
\mathrm{df}(\lambda) = \operatorname{tr}(2H - HH^\top).
$$

The degrees-of-freedom calculation uses the generalized eigenvalues of the
penalty and weighted design pair $(K, X^\top W X)$:

$$
K v_\ell = d_\ell (X^\top W X) v_\ell, \qquad \ell = 1, \dots, p,
$$

where $(d_\ell, v_\ell)$ are the generalized eigenvalue--eigenvector pairs.
These satisfy $v_\ell^\top (X^\top W X) v_m = \delta_{\ell m}$ (orthonormal
with respect to $X^\top W X$) and $v_\ell^\top K v_m = d_\ell \delta_{\ell m}$
when the eigenvectors are properly scaled.

Under this basis, the penalised Gram diagonalises:

$$
G = X^\top W X + \lambda K
  = (X^\top W X)^{1/2} (I + \lambda (X^\top W X)^{-1/2} K (X^\top W X)^{-1/2}) (X^\top W X)^{1/2},
$$

and in the eigenbasis $(v_\ell)$ we have

$$
v_\ell^\top G v_m = v_\ell^\top (X^\top W X) v_m + \lambda v_\ell^\top K v_m
                 = (1 + \lambda d_\ell) \delta_{\ell m}.
$$

Therefore $G^{-1} = \sum_\ell \frac{v_\ell v_\ell^\top}{1 + \lambda d_\ell}$,
and

$$
H = W^{1/2}X G^{-1} X^\top W^{1/2}
  = \sum_\ell \frac{W^{1/2}X v_\ell (W^{1/2}X v_\ell)^\top}{1 + \lambda d_\ell}.
$$

Let $z_\ell=W^{1/2}Xv_\ell$. These vectors are orthonormal because
$z_\ell^\top z_m=v_\ell^\top X^\top W Xv_m=\delta_{\ell m}$.
The traces become:

$$
\operatorname{tr}(H) = \sum_\ell \frac{z_\ell^\top z_\ell}{1 + \lambda d_\ell}
                     = \sum_\ell \frac{1}{1 + \lambda d_\ell},
$$

$$
\operatorname{tr}(HH^\top) = \sum_\ell \frac{1}{(1 + \lambda d_\ell)^2}
                            \cdot (z_\ell^\top z_\ell)^2
                          = \sum_\ell \frac{1}{(1 + \lambda d_\ell)^2}.
$$

Combining,

$$
\boxed{\mathrm{df}(\lambda) = \sum_{\ell=1}^p \left( \frac{2}{1 + \lambda\, d_\ell} - \frac{1}{(1 + \lambda\, d_\ell)^2} \right)}.
$$

Each eigenvalue $d_\ell \ge 0$ contributes to the effective degrees of freedom.
Eigenvalues in the null space of $K$ have $d_\ell = 0$, contributing exactly 1
to $\mathrm{df}(\lambda)$ regardless of $\lambda$ — these are the
**unpenalised** directions (e.g., the linear function for a second-order
difference penalty). Eigenvalues with $d_\ell \to \infty$ contribute 0 as
$\lambda \to \infty$, meaning those directions are fully penalised away.

The function $\mathrm{df}(\lambda)$ is monotone decreasing in $\lambda$:
as $\lambda$ increases, every term shrinks from 1 toward 0 (or stays at 1 for
$d_\ell = 0$). This monotonicity guarantees that the root-find for
$\mathrm{df}^\star$ has a unique solution whenever
$\dim \ker K < \mathrm{df}^\star \le \operatorname{rank}(X)$.

## Solving $\mathrm{df}^\star$ to $\lambda$

Given a target $\mathrm{df}^\star$, the smoothing parameter is the root of

$$
\mathrm{df}(\lambda) - \mathrm{df}^\star = 0,
$$

found by a one-dimensional search. Because $\mathrm{df}(\cdot)$ is smooth and
monotone, a bracketed root-finder (Brent's method) converges quickly. The computation
needs only the eigenstructure of the penalized problem: writing the generalized
eigenvalues $d_1, \dots, d_p$ of $(K, X^\top W X)$, one has the closed form above,
so each evaluation during the root-find is $O(p)$ after a one-time decomposition.
[](../implementation/linear_algebra) describes the implementation (the
spectral pre-factorization and the re-solve per cross-validation fold).

### The root-find in detail

The engine solves the equation with **Brent's method**, the bracketed
root-finder of the Netlib `zeroin` routine. R's `uniroot` calls the same
routine, and R `mboost` calls `uniroot` on the same bracket at the same
tolerance, so the two packages solve this equation by the same algorithm.

1. **The bracket is fixed, not searched for.** It is $[0, \lambda_{\max}]$ with
   $\lambda_{\max} = 10^{15}$. No expansion is needed because the two endpoints
   are the extremes of a monotone function: $\mathrm{df}(0) = \operatorname{rank}(X)$
   and $\mathrm{df}(\lambda_{\max}) \approx \dim\ker K$. Two guards run first —
   $\mathrm{df}^\star \ge \mathrm{df}(0)$ returns $\lambda = 0$, and
   $\mathrm{df}^\star \le \mathrm{df}(\lambda_{\max})$ returns $\lambda_{\max}$
   — so whatever reaches the root-finder is bracketed with a sign change.

2. **Each step is inverse quadratic interpolation, secant, or bisection.**
   Brent's method tries the fastest of the three that stays inside the current
   bracket and shrinks it, falling back to bisection otherwise. No derivative is
   computed: the method needs only function values, and $\mathrm{df}$ is
   $O(p)$ to evaluate once the eigenvalues $d_\ell$ are known.

3. **It stops on the bracket, not on a step count.** The iteration returns when
   the bracket half-width falls below $(\texttt{xtol} + \texttt{rtol}\,|\lambda|)/2$
   with both tolerances at $\sqrt{\varepsilon_{\text{machine}}} \approx 1.5 \times 10^{-8}$,
   R's default. On the data set used for the comparison with R, the result
   agrees with R's $\lambda$ to $2 \times 10^{-7}$ relative.

The search is on the linear $\lambda$ scale, not $\log\lambda$. Bracketed
methods do not need the log rescaling that an unbracketed Newton iteration would,
because bisection bounds the worst case regardless of scale.

The decomposition that produces the generalized eigenvalues $d_\ell$ is a
one-time $O(p^3)$ cost per set of weights. Over the boosting loop it is
amortized over all $m_{\text{stop}}$ iterations; each CV fold has its own
weights, so the decomposition and $\lambda$ are recomputed per fold, and the
fold fits run in parallel.

### Numerical illustration

The df-to-$\lambda$ calibration for a 14-column cosine basis on 50 equally
spaced points with a second-order difference penalty:

In [1]:
import numpy as np
from scipy.linalg import eigh

np.set_printoptions(precision=4, suppress=True)

n, p = 50, 14
rng = np.random.default_rng(1)
x = np.linspace(0, 1, n)
X = np.cos(np.pi * np.arange(p) * x[:, None]) * np.sqrt(2 / n)

# Second-order difference penalty
D = np.zeros((p - 2, p))
for i in range(p - 2):
    D[i, i:i+3] = [1, -2, 1]
K = D.T @ D

# Generalized eigenvalues of (K, XᵀX): returns (w, v) = (eigenvalues, eigenvectors)
d, _ = eigh(K, X.T @ X)
null_dim = int((d < 1e-10).sum())
print(f"Generalized eigenvalues (first 6): {np.round(d[:6], 4)}")
print(f"Null-space dimension (d ≈ 0): {null_dim}  (should be 2 for 2nd-diff)")

# df(λ) from closed-form eigenvalue sum
def df(lam):
    t = lam * d
    return np.sum(2 / (1 + t) - 1 / (1 + t)**2)

print(f"df(0)    = {df(0):.4f}  (rank of X = {p})")
print(f"df(1)    = {df(1):.4f}")
print(f"df(10)   = {df(10):.4f}")
print(f"df(1e6)  = {df(1e6):.4f}  (approaches null-space dim = {null_dim})")

# Geometric bisection to solve df(λ) = 4
target = 4.0
lo, hi = 1e-12, 1e12
for it in range(50):
    lam = np.sqrt(lo * hi)
    if df(lam) > target:
        lo = lam
    else:
        hi = lam
    if hi / lo < 1.001:
        break

lam_final = np.sqrt(lo * hi)
print(f"\nλ for df=4: {lam_final:.4f}  (converged in {it+1} iterations)")
print(f"  df({lam_final:.4f}) = {df(lam_final):.6f}")


Generalized eigenvalues (first 6): [-0.      0.      0.0115  0.0895  0.3435  0.9017]
Null-space dimension (d ≈ 0): 2  (should be 2 for 2nd-diff)
df(0)    = 14.0000  (rank of X = 14)
df(1)    = 7.7637
df(10)   = 4.6366
df(1e6)  = 2.0002  (approaches null-space dim = 2)

λ for df=4: 20.9816  (converged in 16 iterations)
  df(20.9816) = 4.000138


The example uses geometric bisection to find the root. The engine uses Brent's
method to reach the same $\lambda$ with fewer evaluations. Either way the
eigenvalues $d$ are computed once per base learner and set of weights and
reused over all $m_{\text{stop}}$ iterations, so the root-find itself is never
the cost.

## Calibrating by `df` instead of `λ`

- **Comparability across learners.** Boosting selects the learner with the
  smallest residual sum of squares. A rich, lightly penalized learner would
  almost always "win" purely because it is more flexible. Fixing every learner to
  the *same small* `df` (the `bbs` default is 4) means selection reflects signal, not
  flexibility {cite}`hofner2011framework`.

- **Weakness.** Small `df` makes each learner a high-bias, low-variance fit, so
  the per-step increment is small and $m_{\text{stop}}$ controls complexity
  smoothly (see [](functional_gradient_boosting.ipynb)).

- **Interpretability and transfer.** `df` is on a scale practitioners understand
  (a `df`-4 smooth has roughly 4 effective parameters) and is independent
  of the units of $x$, unlike $\lambda$.

## Lower bound on `df`: the penalty null space

A second-order difference penalty leaves a linear function unpenalized, so its
minimum attainable `df` is 2. For `bbs`, a `df` below the null-space dimension
raises an error when the learner is built (no finite $\lambda$ achieves it), and
`df` equal to it gives a warning and the maximum penalty
$\lambda_{\max} = 10^{15}$; `center=True` removes the null space. A `df` above
$\operatorname{rank}(X)$ gives $\lambda = 0$, the unpenalized fit. Equivalently, a P-spline can be shrunk *toward* a line but
never below it.

## Weighted and per-fold recalibration

The `df` to `λ` map depends on the weights $W$. In cross-validation each fold
reweights the observations (zeroing out-of-bag points, possibly with integer
resampling weights), so `λ` must be **recomputed per fold** to keep `df` fixed.
## Exercises

````{admonition} Exercise: Derive $\mathrm{df}(\lambda)$ for scalar ridge
:class: dropdown

Consider a nonzero centered predictor $X$ ($n \times 1$) with unit observation
weights and ridge penalty $\lambda I$. Let $s = X^\top X = \sum_i x_i^2 > 0$.
The hat matrix is

$$
H(\lambda) = X\,(X^\top X + \lambda)^{-1}\,X^\top
            = \frac{XX^\top}{s+\lambda}.
$$

**a)** Write $H(\lambda)$ explicitly as an $n \times n$ matrix.

**b)** Compute $\operatorname{tr}(H)$ and $\operatorname{tr}(HH^\top)$.

**c)** Derive $\mathrm{df}(\lambda) = \operatorname{tr}(2H - HH^\top)$ and
show that

$$
\mathrm{df}(\lambda) = \frac{2s}{s+\lambda} - \frac{s^2}{(s+\lambda)^2}
                     = 1 - \frac{\lambda^2}{(s+\lambda)^2}.
$$

**d)** Verify that $\mathrm{df}(0) = 1$ and
$\mathrm{df}(\lambda) \to 0$ as $\lambda \to \infty$.

**e)** Solve for $\lambda$ given a target $\mathrm{df}^\star$:

$$
\lambda = s\,\frac{\sqrt{1-\mathrm{df}^\star}}
{1-\sqrt{1-\mathrm{df}^\star}}.
$$

Show that $\mathrm{df}^\star$ must be in $(0, 1]$. This is the scalar case of
the closed form used in the chapter.

````

````{admonition} Solution
:class: dropdown

**a)** The $(i,j)$ entry is

$$
H_{ij} = \frac{x_i x_j}{s+\lambda}.
$$

**b)** Since $X^\top X=s$,

$$
\operatorname{tr}(H) = \frac{s}{s+\lambda}.
$$

The matrix is symmetric and rank one. Its square is

$$
H^2 = \frac{X X^\top X X^\top}{(s+\lambda)^2}
    = \frac{sXX^\top}{(s+\lambda)^2},
$$

so

$$
\operatorname{tr}(HH^\top) = \operatorname{tr}(H^2)
= \frac{s^2}{(s+\lambda)^2}.
$$

**c)** Substituting these traces gives

$$
\mathrm{df}(\lambda) = \frac{2s}{s+\lambda}
- \frac{s^2}{(s+\lambda)^2}
= 1 - \frac{\lambda^2}{(s+\lambda)^2}.
$$

**d)** At $\lambda=0$, the fitted rank-one projector has trace 1. As
$\lambda \to \infty$, $\lambda/(s+\lambda) \to 1$, so the degrees of
freedom tend to 0.

**e)** Solving for $\lambda$ gives

$$
\frac{\lambda}{s+\lambda} = \sqrt{1-\mathrm{df}^\star},
\qquad
\lambda = s\,\frac{\sqrt{1-\mathrm{df}^\star}+(1-\mathrm{df}^\star)}
{\mathrm{df}^\star}.
$$

The degrees of freedom range from 1 at $\lambda=0$ toward 0 as $\lambda$
increases, so a finite nonnegative penalty gives a target in $(0,1]$.
````

````{admonition} Exercise: Effective df of a second-order P-spline
:class: dropdown

A cubic B-spline basis with 10 inner knots has $p = 14$ columns (10 inner + 3
degree + 1).  The penalty $K = D_2^\top D_2$ is a second-order difference
matrix of rank $p-2$.  The generalized eigenvalues $d_\ell$ of
$(K, X^\top W X)$ are:

$$
\begin{aligned}
d &= [0, 0, 0.002, 0.015, 0.058, 0.153, 0.324, 0.578, \\
  &\qquad 0.911, 1.306, 1.731, 2.150, 2.534, 2.864].
\end{aligned}
$$

**a)** What are $\mathrm{df}(0)$ and $\lim_{\lambda\to\infty} \mathrm{df}(\lambda)$?
Verify they match $\operatorname{rank}(X)$ and $\dim\ker K$.

**b)** Compute $\mathrm{df}(\lambda)$ for $\lambda = 10$ using the closed-form
eigenvalue sum.

**c)** Given a target $\mathrm{df}^\star = 4$, bracket $\lambda$ and perform
**three iterations** of geometric-bisection root-finding (as in the chapter's
numerical illustration).  Show the bracket and midpoint after each iteration.

**d)** For a new dataset with different weights $W$, would the same $\lambda$
give the same $\mathrm{df}$?  Explain.

````

````{admonition} Solution
:class: dropdown

**a)** At $\lambda = 0$, every term contributes 1, so $\mathrm{df}(0) = p = 14$
(the rank of $X$).  As $\lambda \to \infty$, only the $d_\ell = 0$ terms
contribute (each exactly 1).  There are two zero eigenvalues (the null space
of $D_2$: constant and linear functions), so $\mathrm{df}(\infty) = 2$.
This matches $\dim\ker K$.

**b)** For each $\ell$, compute $f(d_\ell) = 2/(1+10 d_\ell) - 1/(1+10 d_\ell)^2$:

| $d_\ell$ | $1+10 d_\ell$ | term |
|---|---|---|
| 0 | 1.00 | 1.0000 |
| 0 | 1.00 | 1.0000 |
| 0.002 | 1.02 | 0.9996 |
| 0.015 | 1.15 | 0.9830 |
| 0.058 | 1.58 | 0.8652 |
| 0.153 | 2.53 | 0.6343 |
| 0.324 | 4.24 | 0.4161 |
| 0.578 | 6.78 | 0.2732 |
| 0.911 | 10.11 | 0.1880 |
| 1.306 | 14.06 | 0.1372 |
| 1.731 | 18.31 | 0.1062 |
| 2.15 | 22.50 | 0.0869 |
| 2.534 | 26.34 | 0.0745 |
| 2.864 | 29.64 | 0.0663 |

Summing the fourteen terms gives $\mathrm{df}(10)\approx6.8307$.

**c)** Since $\mathrm{df}(10)>4$ and $\mathrm{df}(10^6)<4$, start with
$[\lambda_L,\lambda_R]=[10,10^6]$. At each step use the geometric midpoint
$\lambda_M=\sqrt{\lambda_L\lambda_R}$ and evaluate the full eigenvalue sum.

- **Iteration 1:** $\lambda_M\approx3162.28$ and
  $\mathrm{df}(\lambda_M)\approx2.3156$, so the new bracket is $[10,3162.28]$.
- **Iteration 2:** $\lambda_M\approx177.83$ and
  $\mathrm{df}(\lambda_M)\approx3.7346$, giving $[10,177.83]$.
- **Iteration 3:** $\lambda_M\approx42.17$ and
  $\mathrm{df}(\lambda_M)\approx4.9682$, giving $[42.17,177.83]$.

The root is about $123.70$. Further iterations narrow the bracket to the
requested tolerance; their number depends on that tolerance.

**d)** No — the generalized eigenvalues $d_\ell$ depend on $X^\top W X$.
Different weights $W$ change the Gram matrix, hence change the
eigendecomposition and the function $\mathrm{df}(\lambda)$.  This is why,
in cross-validation, $\lambda$ must be re-solved per fold: the out-of-bag
weights change the effective sample and the Gram matrix.
````

````{admonition} Exercise: $\mathrm{df}$ as a sum over eigenvalues
:class: dropdown

The effective degrees of freedom of a penalised linear smoother is

$$
\mathrm{df}(\lambda) = \sum_{\ell=1}^p \left(
\frac{2}{1 + \lambda d_\ell} - \frac{1}{(1 + \lambda d_\ell)^2}
\right),
$$

where $d_\ell \ge 0$ are the generalized eigenvalues of $(K, X^\top W X)$.

**a)** Define $f(t) = 2/(1+t) - 1/(1+t)^2$ for $t \ge 0$. Show that
$f(0) = 1$, $f(\infty) = 0$, and $f$ is strictly decreasing.

**b)** Show that $\mathrm{df}'(\lambda) < 0$ for all $\lambda > 0$, confirming
monotonicity.  (Recall the derivative of $f$ with respect to $\lambda$ is
$f'(t) \cdot d_\ell$.)

**c)** For $d_\ell = 0$ (a null-space direction of $K$), show that the
contribution to $\mathrm{df}(\lambda)$ is exactly 1 for every $\lambda$.
Interpret this: which directions are never penalised?

**d)** Use (a) and (c) to bound $\mathrm{df}(\lambda)$:

$$
\dim\ker K \;\le\; \mathrm{df}(\lambda) \;\le\; \operatorname{rank}(X).
$$

Explain why a user-specified $\mathrm{df}^\star$ must lie above $\dim\ker K$
and at most at $\operatorname{rank}(X)$.

````

````{admonition} Solution
:class: dropdown

**a)** $f(0) = 2/1 - 1/1 = 1$.  As $t \to \infty$, $f(t) \sim 2/t - 1/t^2 \to 0$.
The derivative:

$$
f'(t) = -\frac{2}{(1+t)^2} + \frac{2}{(1+t)^3}
      = -\frac{2t}{(1+t)^3} \le 0,
$$

with strict inequality for $t > 0$.  Hence $f$ is strictly decreasing.

**b)** By the chain rule,

$$
\mathrm{df}'(\lambda) = \sum_\ell f'(\lambda d_\ell) \cdot d_\ell
= -\sum_\ell \frac{2 \lambda d_\ell^2}{(1 + \lambda d_\ell)^3} \le 0.
$$

Since at least one $d_\ell > 0$ whenever $K$ is not the zero matrix,
$\mathrm{df}'(\lambda) < 0$ for $\lambda > 0$.

**c)** When $d_\ell = 0$, $f(\lambda \cdot 0) = f(0) = 1$ for any $\lambda$.
These directions correspond to $\ker K$ — the subspace of the parameter
vector that the penalty leaves unchanged.  For a second-order difference
penalty, $\ker K$ is the set of linear functions (constant + linear term),
which are never shrunk toward zero.

**d)** Each term satisfies $0 \le f(\lambda d_\ell) \le 1$, with equality
at 1 for $d_\ell = 0$ and at 0 only in the limit $d_\ell \to \infty$.
Summing over all $p$ terms:

- **Lower bound:** the null-space terms ($d_\ell = 0$) always contribute 1;
  all other terms contribute $\ge 0$.  Hence $\mathrm{df}(\lambda) \ge |\{d_\ell = 0\}| = \dim\ker K$.
- **Upper bound:** every term is $\le 1$, so $\mathrm{df}(\lambda) \le p$.
  At $\lambda = 0$, every term equals 1, giving $\mathrm{df}(0) = p = \operatorname{rank}(X)$
  (assuming $X$ has full column rank).

If a user requests $\mathrm{df}^\star \le \dim\ker K$, no finite $\lambda$
achieves it: for `bbs` a request below $\dim\ker K$ raises an error, and a
request equal to it is fitted at $\lambda_{\max} = 10^{15}$ with a warning. If
$\mathrm{df}^\star > \operatorname{rank}(X)$, the target is infeasible
(negative $\lambda$ would be required); the engine uses $\lambda = 0$, the
unpenalized fit.
````


## Numerical check: selection bias from unequal `df`

Giving one learner more `df` than another biases selection toward it
regardless of signal {cite}`hofner2011framework`. In the simulation, $n = 300$,
$y = 1 + 0.15\,x_1 + \varepsilon$ with $x_1, x_2, \varepsilon$ independent
standard normal, so `x1` carries a weak signal and `x2` none. The model
`bbs(x1, df=4) + bbs(x2, df=...)` is boosted for 20 iterations ($\nu = 0.1$)
once with `df=4` for `x2` and once with `df=20`, and the number of iterations
that select each column is counted.


In [2]:
import numpy as np
import mboostrs as mb
from collections import Counter

rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(size=n)  # weak but real signal
x2 = rng.normal(size=n)  # pure noise, no relationship to y at all
y = 1.0 + 0.15 * x1 + rng.normal(size=n) * 1.0
data = {"y": y, "x1": x1, "x2": x2}

noise_share = {}
for df2 in [4, 20]:
    fit = mb.gamboost(f"y ~ bbs(x1, df=4) + bbs(x2, df={df2})", data=data,
                       family=mb.Gaussian(), control=mb.boost_control(mstop=20, nu=0.1))
    counts = Counter(fit.selected[:20])
    noise_share[df2] = counts.get("x2", 0) / 20
    print(f"df(x1)=4, df(x2)={df2:>2}: x1 selected {counts.get('x1',0):2d}/20, "
          f"x2 (pure noise) selected {counts.get('x2',0):2d}/20")

assert noise_share[20] > noise_share[4], (
    "giving the noise column more df should make it win more often, not less"
)
print(f"\nnoise column's selection share: {noise_share[4]:.2f} (equal df) "
      f"-> {noise_share[20]:.2f} (df=20) -- more df wins more, regardless of signal")


df(x1)=4, df(x2)= 4: x1 selected 11/20, x2 (pure noise) selected  9/20
df(x1)=4, df(x2)=20: x1 selected  8/20, x2 (pure noise) selected 12/20

noise column's selection share: 0.45 (equal df) -> 0.60 (df=20) -- more df wins more, regardless of signal


With equal `df` the noise column `x2` is selected in 9 of the 20 iterations;
with `df=20` for `x2` it is selected in 12. The extra flexibility, not a
signal, raises its share from 0.45 to 0.60.

## Summary

- The effective degrees of freedom of a penalized learner is
  $\mathrm{df}(\lambda) = \operatorname{tr}(2H_\lambda - H_\lambda H_\lambda^\top)$,
  where $H_\lambda$ is the hat matrix of the penalized fit. This is computed
  from the generalized eigenvalues of $(K, X^\top W X)$.
- The user specifies a target $\mathrm{df}^\star$ and the engine solves for
  $\lambda$ by root-finding on the monotone $\mathrm{df}(\lambda)$ curve.
  For a single penalized column without an unpenalized intercept, let
  $s=x^\top W x>0$. The idealized closed form is
  $\lambda=s\sqrt{1-\mathrm{df}^\star}/(1-\sqrt{1-\mathrm{df}^\star})$,
  for $0<\mathrm{df}^\star\le1$. An unpenalized intercept contributes one
  degree of freedom, which is subtracted before calibrating the penalized
  column. The engine adds $10^{-9} K$ to $X^\top W X$ before the
  decomposition, which keeps it invertible when a basis column has almost no
  data, and caps $\lambda$ at no more than $\lambda_{\max} = 10^{15}$.
- The $\lambda$-to-df map depends on the weights $W$. In CV, each fold
  reweights observations, so $\lambda$ must be re-solved per fold — a common
  source of discrepancy if skipped.

## See also

- [](base_learners.ipynb) — the learner catalogue.
- [](../usage/base_learners) — runnable examples with different `df` settings.
- [](../implementation/linear_algebra) — how the root-find is implemented.